<a href="https://colab.research.google.com/github/COMP3608-Group-12/Project/blob/Dataset-3-Decision-Tree-Model/Group_12_Dataset_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 1: Importing Packages

In [1]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

## STEP 2: Load & Understand Data

In [2]:
#MUST BE RUN inorder for file download to work
!pip install gdown

#Downloading dataset csv into runtime from google drive link
import gdown

#train dataset
url = "https://drive.google.com/uc?id=1WxiA2-DJuTOcykkqqTX-LECs9XltPHXt"
gdown.download(url, "train.csv", quiet=False)

#test dataset
url = "https://drive.google.com/uc?id=10qbFPqH3JYWfGmDSaal7IGzfGLb86HQV"
gdown.download(url, "test.csv", quiet=False)

#loading csv files into pandas dataframes
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

Downloading...
From (original): https://drive.google.com/uc?id=1WxiA2-DJuTOcykkqqTX-LECs9XltPHXt
From (redirected): https://drive.google.com/uc?id=1WxiA2-DJuTOcykkqqTX-LECs9XltPHXt&confirm=t&uuid=6dbaa8d3-ced1-4e2c-9c3a-35bf10428f63
To: /content/train.csv
100%|██████████| 351M/351M [00:05<00:00, 69.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=10qbFPqH3JYWfGmDSaal7IGzfGLb86HQV
From (redirected): https://drive.google.com/uc?id=10qbFPqH3JYWfGmDSaal7IGzfGLb86HQV&confirm=t&uuid=786051ad-e5a5-49df-b5be-4e4dcef63852
To: /content/test.csv
100%|██████████| 150M/150M [00:02<00:00, 60.1MB/s]


In [3]:
#printing dataset information (how many rows, columns, datatype etc)
print(train_df.info())
print('----------------------------------------------------------')

#checking if any values are missing
print(train_df.isnull().sum())
print('----------------------------------------------------------')

# Show ONLY columns with missing values
print("Columns with missing values:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])
print('----------------------------------------------------------')

#Dropping rows with missing values, if any
train_df = train_df.dropna()

#Verifying that those rows have been dropped and there are no missing values
print("Total missing values after dropping:")
print(train_df.isnull().sum().sum())  # should be 0
print('----------------------------------------------------------')

#printing first 5 rows of data to ensure it correlates with datafile
print(train_df.head(5))
print('----------------------------------------------------------')

#checking how many transactions are fraudulent and how many are non-fraudulent
print(train_df['is_fraud'].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

##Step 3: Data Preprocessing


###Step 3a: Feature Engineering



In [4]:
from sklearn.preprocessing import LabelEncoder

def build_features(df, fit_encoders=True, encoders=None):
    df = df.copy()

    # Converting fields with date & time from object to datetime
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob'] = pd.to_datetime(df['dob'])

    # Extracting individual features from trans_date_trans_time feature
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['dayofweek'] = df['trans_date_trans_time'].dt.dayofweek
    df['month'] = df['trans_date_trans_time'].dt.month

    # Converting DOB to age
    df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days

    # Label Encoding remaining columns of type object
    cat_cols = ['merchant', 'category', 'gender', 'state', 'job']

    if fit_encoders:
        encoders = {}
        for col in cat_cols:
            le = LabelEncoder()
            values = df[col].astype(str).tolist()
            values.append('unknown')
            le.fit(values)
            df[col + '_enc'] = le.transform(df[col].astype(str))
            encoders[col] = le
    else:
        for col in cat_cols:
            le = encoders[col]
            known = set(le.classes_)

            safe_values = np.where(
                df[col].astype(str).isin(known),
                df[col].astype(str),
                'unknown'
            )

            df[col + '_enc'] = le.transform(safe_values)

    return df, encoders

###Step 3b: Preprocessing on train data

In [5]:
# Apply feature engineering to test_df
train_df, encoders = build_features(train_df, fit_encoders=True)

# Drop unnecessary columns
train_df = train_df.drop(columns=[
    'Unnamed: 0', 'first', 'last', 'street',
    'trans_num', 'trans_date_trans_time', 'dob'
])

# Separate features and target
X = train_df.drop('is_fraud', axis=1)
y = train_df['is_fraud']

# Select only numerical columns for X
X = X.select_dtypes(include=['int64', 'float64'])

###Step 3c: Preprocessing on test data

In [6]:
# Apply feature engineering to test_df
test_df, _ = build_features(test_df, fit_encoders=False, encoders=encoders)

# Drop the same columns as in train_df
test_df = test_df.drop(columns=[
    'Unnamed: 0', 'first', 'last', 'street',
    'trans_num', 'trans_date_trans_time', 'dob'
])

# Separate features and target
X_test_final = test_df.drop('is_fraud', axis=1)
y_test_final = test_df['is_fraud']

# Select only numerical columns for X_test_final, consistent with how X was created
X_test_final = X_test_final.select_dtypes(include=['int64', 'float64'])

##Step 4: Decision Tree Model using KFold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Decision Tree Model
model_dt = DecisionTreeClassifier(max_depth=10, random_state=42)

# StratifiedKFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
dt_fold_results = []

# Manual KFold Loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):

    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Train model
    model_dt.fit(X_train_resampled, y_train_resampled)

    # Predict on test data
    y_pred = model_dt.predict(X_test)
    y_prob = model_dt.predict_proba(X_test)[:, 1]

    #Performance metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df = pd.DataFrame({
        "Metric": list(results.keys()),
        "Value": [round(value, 4) for value in results.values()]
    })

    print(results_df)

    print(f"\n============================================")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print(f"\n============================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Appending results
    dt_fold_results.append(results)

# Taking average of all results
dt_results_df = pd.DataFrame(dt_fold_results)

avg_results_dt = dt_results_df.mean()
std_results_dt = dt_results_df.std()

final_results_dt = pd.DataFrame({
    "Metric": avg_results_dt.index,
    "Average Value": [round(val, 4) for val in avg_results_dt.values],
    "Std Dev": [round(val, 4) for val in std_results_dt.values]
})

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_results_dt)


================== Fold 1 ==================
      Metric   Value
0   Accuracy  0.9567
1  Precision  0.1069
2     Recall  0.8802
3   F1 Score  0.1907
4    ROC-AUC  0.9537

Confusion Matrix:
[[123397   5520]
 [    90    661]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9993    0.9572    0.9778    128917
           1     0.1069    0.8802    0.1907       751

    accuracy                         0.9567    129668
   macro avg     0.5531    0.9187    0.5842    129668
weighted avg     0.9941    0.9567    0.9732    129668


================== Fold 2 ==================
      Metric   Value
0   Accuracy  0.9571
1  Precision  0.1110
2     Recall  0.9148
3   F1 Score  0.1980
4    ROC-AUC  0.9729

Confusion Matrix:
[[123414   5503]
 [    64    687]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9995    0.9573    0.9779    128917
           1     0.1110    0.9148    0.1980       751

    accur